In [20]:
!pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable


In [21]:
%pip install python-dotenv 
import os

from dotenv import load_dotenv
load_dotenv()

os.environ["langsmith_api_key"]=os.getenv("langsmith_api_key")
os.environ["langchain-groq"]=os.getenv("langchain-groq")
os.environ["langsmith_tracing"]="true"


Note: you may need to restart the kernel to use updated packages.


In [22]:
%pip install langsmith
from langsmith import Client
client=Client()

# Change to a new, unique name
dataset_name = "my-dataset-v2" 
dataset = client.create_dataset(dataset_name=dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples = [
    {
        "inputs": {"question": "What is Hugging Face?"},
        "outputs": {"answer": "A platform and community for sharing machine learning models and datasets"},
    },
    {
        "inputs": {"question": "What is RAG?"},
        "outputs": {"answer": "A technique that enhances LLMs by retrieving relevant information from external sources"},
    },
    {
        "inputs": {"question": "What is Prompt Engineering?"},
        "outputs": {"answer": "The process of designing and refining inputs to get the best possible output from an AI model"},
    },
    {
        "inputs": {"question": "What is a Vector Database?"},
        "outputs": {"answer": "A specialized database designed to store and query high-dimensional vector embeddings"},
    },
    {
        "inputs": {"question": "What is fine-tuning?"},
        "outputs": {"answer": "Adjusting a pre-trained model on a smaller, specific dataset to improve its performance on a particular task"},
    },
    {
        "inputs": {"question": "What is AI hallucination?"},
        "outputs": {"answer": "When a language model generates false or nonsensical information but presents it as fact"},
    },
    {
        "inputs": {"question": "What is tokenization?"},
        "outputs": {"answer": "The process of breaking text down into smaller pieces that a machine learning model can process"},
    },
    {
        "inputs": {"question": "What is an embedding?"},
        "outputs": {"answer": "A numerical representation of text or data that captures its semantic meaning"},
    },
    {
        "inputs": {"question": "What is Llama 3?"},
        "outputs": {"answer": "An open-weights large language model developed by Meta"},
    },
    {
        "inputs": {"question": "What is LangServe?"},
        "outputs": {"answer": "A tool that helps developers deploy LangChain chains and runnables as REST APIs"},
    }
]
)



  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
  Attempting uninstall: h11
    Found existing installation: h11 0.14.0
    Uninstalling h11-0.14.0:
      Successfully uninstalled h11-0.14.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpcore 1.0.7 requires h11<0.15,>=0.13, but you have h11 0.16.0 which is incompatible.


LangSmithConflictError: Conflict for /datasets. HTTPError('409 Client Error: Conflict for url: https://api.smith.langchain.com/datasets', '{"detail":"Dataset with this name already exists."}')

In [33]:
%pip install groq
import os
from groq import Groq
from langsmith import traceable

# 1. Initialize the official native Groq client
client = Groq(api_key=os.environ.get("langchain-groq"))

eval_instructions = "You are an expert professor specialized in grading students' answers to question"

# 2. Add @traceable so LangSmith tracks this function
@traceable(run_type="llm", name="Correctness Evaluator")
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    
    user_content = f"""You are grading the following question:
{inputs['question']}
Here is the real answer:
{reference_outputs['answer']}
You are grading the following predicted answer:
{outputs['response']}
Respond with CORRECT or INCORRECT:
Grade:
"""
    
    # 3. Call Groq natively
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant", # Use your preferred Groq model
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content}
        ],
        temperature=0.0 
    )
    
    # 4. Parse the text response into a boolean
    grade = response.choices[0].message.content.strip().upper()
    return "CORRECT" in grade

  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
Using cached h11-0.14.0-py3-none-any.whl (58 kB)
  Attempting uninstall: h11
    Found existing installation: h11 0.16.0
    Uninstalling h11-0.16.0:
      Successfully uninstalled h11-0.16.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpcore2 2.13.1 requires h11>=0.16, but you have h11 0.14.0 which is incompatible.


In [34]:
## Concisions- checks whether the actual output is less than 2x the length of the expected

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

In [43]:
from langsmith import traceable

@traceable(run_type="llm", name="Correctness Evaluator")
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
{inputs['question']}
Here is the real answer:
{reference_outputs['answer']}
You are grading the following predicted answer:
{outputs['response']}
Respond with CORRECT or INCORRECT:
Grade:
"""
    
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile", # <-- Updated model here
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content}
        ],
        temperature=0.0 
    )
    
    grade = response.choices[0].message.content.strip().upper()
    return "CORRECT" in grade

In [44]:
def ls_target(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"])}

In [45]:
from langsmith import Client

# Initialize the LangSmith client (using a distinct variable name)
ls_client = Client()

# Run the evaluation using the LangSmith client
experiment_results = ls_client.evaluate(
    ls_target, 
    data="my-dataset-v2", 
    evaluators=[correctness, concision], 
    experiment_prefix="llama-3.3-70b-versatile"
)

Error running target function: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
Traceback (most recent call last):
  File "c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\langsmith\evaluation\_runner.py", line 2062, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
    ~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\langsmith\run_helpers.py", line 850, in wrapper
    function_result = run_container["context"].run(
        func, *args, **kwargs
    )
  File "C:\Users\hp\AppData\Local\Temp\ipykernel_20656\3475679084.py", line 2, in ls_target
    return {"response": my_app(inputs["question"])}
                        ~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\hp\AppData\Local\Temp\ipykernel_20656\3713507770.py", line 11, in my_app
    return groq_cl

View the evaluation results for experiment: 'llama-3.3-70b-versatile-8e121515' at:
https://smith.langchain.com/o/3f2830bf-3acc-4fe0-944a-b5c3672c3da5/datasets/8c60364b-2c2c-4be1-9ee4-6fcdcee5cc6d/compare?selectedSessions=8cc56ae5-aa5c-4a87-8132-0cf79afc1689




Error running evaluator <DynamicRunEvaluator correctness> on run 01a0f961-70d2-7f83-a4e3-db3418333c2d: KeyError('response')
Traceback (most recent call last):
  File "c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\langsmith\evaluation\_runner.py", line 1723, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
        run=run,
        example=example,
        evaluator_run_id=evaluator_run_id,
    )
  File "c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\langsmith\evaluation\evaluator.py", line 370, in evaluate_run
    result = self.func(
        run,
        example,
        langsmith_extra={"run_id": evaluator_run_id, "metadata": metadata},
    )
  File "c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\langsmith\run_helpers.py", line 850, in wrapper
    function_result = run_container["context"].run(
        func, *args, **kwargs
    )
  File "c:\Users\hp\AppData\Local\Programs\

In [ ]:

import requests

# 1. Paste your exact API key here (starts with 'gsk

url = "https://api.groq.com/openai/v1/models"
headers = {
    "Authorization": f"Bearer {my_groq_key}",
    "Content-Type": "application/json"
}
response = requests.get(url, headers=headers)

# 2. Print the exact response to see if it connects or fails
print(f"Status Code: {response.status_code}")
print(response.json())

Status Code: 200
{'object': 'list', 'data': [{'id': 'openai/gpt-oss-20b', 'object': 'model', 'created': 1754407957, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536, 'hugging_face_id': 'openai/gpt-oss-20b', 'name': 'GPT OSS 20B', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 131072, 'max_output_length': 65536, 'pricing': {'prompt': '0.000000075', 'completion': '0.0000003', 'image': '0', 'request': '0', 'input_cache_read': '0.0000000375'}, 'supported_sampling_parameters': ['temperature', 'top_p', 'stop', 'seed', 'max_tokens'], 'supported_features': ['tools', 'json_mode', 'structured_outputs', 'reasoning']}, {'id': 'allam-2-7b', 'object': 'model', 'created': 1737672203, 'owned_by': 'SDAIA', 'active': True, 'context_window': 4096, 'public_apps': None, 'max_completion_tokens': 4096, 'hugging_face_id': 'ALLaM-AI/ALLaM-2.0-7B-Instruct', 'name': 'ALLaM-2-7b', 'input_modalities': ['text'], 'outp

In [51]:
def my_app(question: str, model: str = "openai/gpt-oss-20b", instructions: str = default_instructions) -> str:
    return groq_client.chat.completions.create(  
        model=model, 
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [52]:
from langsmith import traceable

@traceable(run_type="llm", name="Correctness Evaluator")
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
{inputs['question']}
Here is the real answer:
{reference_outputs['answer']}
You are grading the following predicted answer:
{outputs['response']}
Respond with CORRECT or INCORRECT:
Grade:
"""
    
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b", # <-- Using your available model
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content}
        ],
        temperature=0.0 
    )
    
    grade = response.choices[0].message.content.strip().upper()
    return "CORRECT" in grade

In [53]:
from langsmith import Client

# Initialize the LangSmith client (using a distinct variable name)
ls_client = Client()

# Run the evaluation using the LangSmith client
experiment_results = ls_client.evaluate(
    ls_target, 
    data="my-dataset-v2", 
    evaluators=[correctness, concision], 
    experiment_prefix="llama-3.3-70b-versatile"
)

View the evaluation results for experiment: 'llama-3.3-70b-versatile-bd778c57' at:
https://smith.langchain.com/o/3f2830bf-3acc-4fe0-944a-b5c3672c3da5/datasets/8c60364b-2c2c-4be1-9ee4-6fcdcee5cc6d/compare?selectedSessions=8d91a33f-91ae-469d-b1f3-b961aeeb3e64




In [56]:
## RAG with Groq / Local Embeddings
%pip install langchain_community langchain_core langchain_huggingface langchain_text_splitters sentence-transformers

from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

# Initialize local Hugging Face embeddings (runs entirely on your machine, no OpenAI needed)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Add the document chunks to the vector store using local embeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embeddings,
)

# Turn your vector store into a retrieval component
retriever = vectorstore.as_retriever(k=6)

  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
  Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached scikit_learn-1.9.1-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp313-cp313-win_amd64.whl.metadata (61 kB)
  Using cached typer-0.27.2-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached setuptools-84.0.0-py3-none-a

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [57]:
retriever.invoke("what is agents")

[Document(id='5c606bc7-56dc-4383-a26a-8b203bc3800a', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [59]:
%pip install langchain
from langchain.chat_models import init_chat_model
llm=init_chat_model("openai:gpt-4o-mini")
llm


  Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached ormsgpack-1.12.2-cp313-cp313-win_amd64.whl.metadata (3.3 kB)
Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl (56 kB)
Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl (41 kB)
Using cached ormsgpack-1.12.2-cp313-cp313-win_amd64.whl (117 kB)

  Attempting uninstall: websockets

    Found existing installation: websockets 17.1

    Uninstalling websockets-17.1:

      Successfully uninstalled websockets-17.1

   ---------------------------------------- 0/7 [websockets]
   ---------------------------------------- 0/7 [websockets]
   ---------------------------------------- 0/7 [websockets]
   ---------------------------------------- 0/7 [websockets]
   ---------------------------------------- 0/7 [websockets]
   ---------------------------------------- 0/7 [websockets]
   ---------------------------------------

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.